In [29]:
# =====================================================================
# 03-04 - SABİT ENSEMBLE SEÇİMİ İLE 2025-2030 GELECEK TAHMİNİ
#
# Bu kod artık 2002-2024 üzerinde model karşılaştırması / fold testi yapmaz.
# Senin paylaştığın TOP-K model ve ağırlıkları doğrudan kullanır.
#
# Akış:
#   1) model_giris.csv dosyasından 2002-2024 gerçek sağlık verisini okur.
#   2) TÜİK nüfus projeksiyon Excel dosyasından 2025-2030 nüfusu okur.
#   3) Aşağıda sabit verilen ensemble model-ağırlık listesini kullanır.
#   4) Sadece seçilmiş modelleri 2002-2024 verisine fit eder.
#      Not: Bu yeni bir performans denemesi değildir; gelecek tahmini için zorunlu model fit işlemidir.
#   5) 2025-2030 için il bazlı tahmin üretir.
#
# Gerekli dosyalar:
#   - model_giris.csv
#   - hamveriler\tuik26-30hesaplanmis\Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx
#
# Çıktılar:
#   - ensemble_secimleri_sabit.csv
#   - nufus_projeksiyonu_2025_2030.csv
#   - gelecek_baz_ensemble_sabit.csv
#   - tahmin_2025_2030_ensemble_sabit.csv
#   - tahmin_2025_2030_ensemble_sabit.xlsx
# =====================================================================

import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
except ImportError:
    HAVE_CAT = False

In [31]:
# %% [1] PARAMETRELER =================================================
MODEL_GIRIS_YOL = "model_giris.csv"
TUIK_YOL = r"hamveriler\tuik26-30hesaplanmis\Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"

# Dosya aynı klasördeyse otomatik bulsun
if not os.path.exists(TUIK_YOL) and os.path.exists("Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"):
    TUIK_YOL = "Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"

BASLANGIC_YIL = 2025
BITIS_YIL = 2030
GELECEK_YILLAR = list(range(BASLANGIC_YIL, BITIS_YIL + 1))

RANDOM_STATE = 42
ITERASYON_TURU = 3

HEDEFLER = ["doktor", "ebe", "diger", "hemsire", "yatak", "hastane"]

FEATURE_MAP = {
    "doktor":  ["yıl", "nufus", "hastane", "yatak", "hemsire", "ebe", "diger"],
    "ebe":     ["yıl", "nufus", "hastane", "yatak", "doktor", "hemsire", "diger"],
    "diger":   ["yıl", "nufus", "hastane", "yatak", "doktor", "hemsire", "ebe"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "doktor", "ebe", "diger"],
    "yatak":   ["yıl", "nufus", "hastane", "doktor", "hemsire", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "doktor", "hemsire", "ebe", "diger"],
}

UNIVARIATE_MODELS = {"ARIMA", "HoltWinters"}

In [33]:
# %% [2] SABİT ENSEMBLE SEÇİMİ =======================================
# Bu tablo senin paylaştığın Fold-5 TOP-K model ve ağırlık sonuçlarından alınmıştır.
SABIT_SECIMLER = [
    # diger
    {"Hedef": "diger", "Model": "ARIMA", "Accuracy": 90.598394, "MAE": 375.372619, "RMSE": 1146.702571, "Agirlik": 0.336859},
    {"Hedef": "diger", "Model": "HoltWinters", "Accuracy": 90.478957, "MAE": 379.220052, "RMSE": 1206.847747, "Agirlik": 0.336415},
    {"Hedef": "diger", "Model": "HistGradientBoosting", "Accuracy": 87.873100, "MAE": 607.133129, "RMSE": 2742.472716, "Agirlik": 0.326726},

    # doktor
    {"Hedef": "doktor", "Model": "HoltWinters", "Accuracy": 91.975715, "MAE": 231.135847, "RMSE": 685.106223, "Agirlik": 0.337337},
    {"Hedef": "doktor", "Model": "ARIMA", "Accuracy": 90.619329, "MAE": 300.440868, "RMSE": 947.615696, "Agirlik": 0.332362},
    {"Hedef": "doktor", "Model": "CatBoost", "Accuracy": 90.057644, "MAE": 283.047003, "RMSE": 1082.125630, "Agirlik": 0.330302},

    # ebe
    {"Hedef": "ebe", "Model": "ARIMA", "Accuracy": 93.776671, "MAE": 34.045168, "RMSE": 68.273248, "Agirlik": 0.345603},
    {"Hedef": "ebe", "Model": "HoltWinters", "Accuracy": 92.300293, "MAE": 39.515053, "RMSE": 64.821849, "Agirlik": 0.340162},
    {"Hedef": "ebe", "Model": "RandomForest", "Accuracy": 85.264967, "MAE": 82.050586, "RMSE": 127.816354, "Agirlik": 0.314234},

    # hastane
    {"Hedef": "hastane", "Model": "ARIMA", "Accuracy": 95.956145, "MAE": 0.654135, "RMSE": 1.411002, "Agirlik": 0.356412},
    {"Hedef": "hastane", "Model": "HoltWinters", "Accuracy": 93.197268, "MAE": 0.974772, "RMSE": 1.572279, "Agirlik": 0.346165},
    {"Hedef": "hastane", "Model": "Voting", "Accuracy": 80.074442, "MAE": 2.558210, "RMSE": 5.837821, "Agirlik": 0.297423},

    # hemsire
    {"Hedef": "hemsire", "Model": "ExtraTrees", "Accuracy": 93.937346, "MAE": 166.520514, "RMSE": 421.617079, "Agirlik": 0.334710},
    {"Hedef": "hemsire", "Model": "Voting", "Accuracy": 93.531235, "MAE": 198.314161, "RMSE": 551.330462, "Agirlik": 0.333263},
    {"Hedef": "hemsire", "Model": "XGBoost", "Accuracy": 93.184293, "MAE": 219.762991, "RMSE": 624.091181, "Agirlik": 0.332027},

    # yatak
    {"Hedef": "yatak", "Model": "ARIMA", "Accuracy": 92.710238, "MAE": 264.545726, "RMSE": 837.159518, "Agirlik": 0.341418},
    {"Hedef": "yatak", "Model": "HoltWinters", "Accuracy": 92.032737, "MAE": 263.946537, "RMSE": 882.777760, "Agirlik": 0.338923},
    {"Hedef": "yatak", "Model": "CatBoost", "Accuracy": 86.801386, "MAE": 273.434879, "RMSE": 530.160839, "Agirlik": 0.319658},
]

In [35]:
# %% [3] YARDIMCI FONKSİYONLAR =======================================
def norm_il(x):
    return str(x).strip()


def read_population_projection(path):
    """Senin Excel dosyana göre: sheet='ML Formatı', header=1, kolonlar: Yıl, İl, Nüfus."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Nüfus projeksiyon dosyası bulunamadı: {path}")

    nf = pd.read_excel(path, sheet_name="ML Formatı", header=1)
    nf.columns = [str(c).strip() for c in nf.columns]

    gerekli = ["Yıl", "İl", "Nüfus"]
    eksik = [c for c in gerekli if c not in nf.columns]
    if eksik:
        raise ValueError(f"Nüfus Excel dosyasında beklenen kolonlar yok: {eksik}. Mevcut kolonlar: {nf.columns.tolist()}")

    nf = nf[["Yıl", "İl", "Nüfus"]].copy()
    nf.columns = ["yıl", "il_adı", "nufus"]
    nf["il_adı"] = nf["il_adı"].apply(norm_il)
    nf["yıl"] = pd.to_numeric(nf["yıl"], errors="coerce")
    nf["nufus"] = pd.to_numeric(nf["nufus"], errors="coerce")

    nf = nf[nf["yıl"].between(BASLANGIC_YIL, BITIS_YIL)].dropna(subset=["il_adı", "yıl", "nufus"])
    nf["yıl"] = nf["yıl"].astype(int)
    nf["nufus"] = nf["nufus"].astype(float)
    nf = nf.groupby(["il_adı", "yıl"], as_index=False)["nufus"].mean()
    return nf


def fc_holtwinters(seri, h):
    seri = pd.Series(seri).astype(float).dropna()
    if len(seri) == 0:
        return np.repeat(np.nan, h)
    if len(seri) < 4 or seri.std() == 0:
        return np.repeat(float(seri.iloc[-1]), h)
    try:
        model = ExponentialSmoothing(seri, trend="add", seasonal=None).fit(optimized=True)
        return np.clip(model.forecast(steps=h).values, 0, None)
    except Exception:
        return np.repeat(float(seri.iloc[-1]), h)


def fc_arima(seri, h):
    seri = np.asarray(pd.Series(seri).astype(float).dropna(), dtype=float)
    if len(seri) == 0:
        return np.repeat(np.nan, h)
    if len(seri) < 6:
        return np.repeat(float(seri[-1]), h)

    best_model, best_aic = None, np.inf
    for order in [(1, 1, 1), (2, 1, 1), (0, 1, 1), (1, 1, 2)]:
        try:
            model = ARIMA(seri, order=order).fit()
            if model.aic < best_aic:
                best_model, best_aic = model, model.aic
        except Exception:
            continue

    if best_model is None:
        return np.repeat(float(seri[-1]), h)
    return np.clip(np.asarray(best_model.forecast(steps=h)), 0, None)


def univariate_forecast(model_ad, years, values, h):
    if model_ad == "HoltWinters":
        return fc_holtwinters(values, h)
    if model_ad == "ARIMA":
        return fc_arima(values, h)
    raise ValueError(f"Bilinmeyen univariate model: {model_ad}")


def make_model(model_ad):
    if model_ad == "LinearRegression":
        return LinearRegression()
    if model_ad == "RandomForest":
        return RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    if model_ad == "ExtraTrees":
        return ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    if model_ad == "GradientBoosting":
        return GradientBoostingRegressor(n_estimators=300, random_state=RANDOM_STATE)
    if model_ad == "HistGradientBoosting":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    if model_ad == "XGBoost":
        if not HAVE_XGB:
            raise ImportError("XGBoost yüklü değil. Kurulum: pip install xgboost")
        return XGBRegressor(
            n_estimators=400,
            random_state=RANDOM_STATE,
            verbosity=0,
            objective="reg:squarederror",
        )
    if model_ad == "CatBoost":
        if not HAVE_CAT:
            raise ImportError("CatBoost yüklü değil. Kurulum: pip install catboost")
        return CatBoostRegressor(iterations=400, depth=6, verbose=0, random_seed=RANDOM_STATE)
    if model_ad == "SVR":
        return make_pipeline(StandardScaler(), SVR(C=10, epsilon=0.1))
    if model_ad == "KNN":
        return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
    if model_ad == "Voting":
        estimators = [
            ("rf", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
            ("gb", GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE)),
        ]
        if HAVE_XGB:
            estimators.append(("xgb", XGBRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
                verbosity=0,
                objective="reg:squarederror",
            )))
        return VotingRegressor(estimators)

    raise ValueError(f"Bilinmeyen model adı: {model_ad}")


def weighted_average(pred_dict, weight_dict):
    vals, weights = [], []
    for model_ad, pred in pred_dict.items():
        if pred is None or pd.isna(pred):
            continue
        vals.append(float(pred))
        weights.append(float(weight_dict.get(model_ad, 0)))

    if not vals or sum(weights) == 0:
        return np.nan

    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    return float(np.dot(np.asarray(vals, dtype=float), weights))


def check_required_columns(df, cols, name):
    eksik = [c for c in cols if c not in df.columns]
    if eksik:
        raise ValueError(f"{name} içinde eksik kolonlar var: {eksik}. Mevcut kolonlar: {df.columns.tolist()}")

In [37]:
# %% [4] VERİ OKUMA ===================================================
t0 = time.time()
print("========== 1) VERİ OKUMA ==========")

if not os.path.exists(MODEL_GIRIS_YOL):
    raise FileNotFoundError(f"Model giriş dosyası bulunamadı: {MODEL_GIRIS_YOL}")

df = pd.read_csv(MODEL_GIRIS_YOL)
df.columns = [str(c).strip() for c in df.columns]

# doktor yoksa uzman + pratisyen + asistan toplamından oluştur
if "doktor" not in df.columns:
    doktor_alt = ["uzman", "pratisyen", "asistan"]
    check_required_columns(df, doktor_alt, "model_giris.csv")
    df["doktor"] = df["uzman"] + df["pratisyen"] + df["asistan"]

check_required_columns(df, ["il_adı", "yıl", "nufus"] + HEDEFLER, "model_giris.csv")

df["il_adı"] = df["il_adı"].apply(norm_il)
df["yıl"] = pd.to_numeric(df["yıl"], errors="coerce").astype(int)
for c in ["nufus"] + HEDEFLER:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["il_adı", "yıl"]).reset_index(drop=True)

print("Model verisi:", df.shape)
print("İl sayısı:", df["il_adı"].nunique())
print("Yıl aralığı:", int(df["yıl"].min()), "→", int(df["yıl"].max()))

nufus_future = read_population_projection(TUIK_YOL)
print("Nüfus projeksiyonu:", nufus_future.shape)
print("Nüfus yıl aralığı:", int(nufus_future["yıl"].min()), "→", int(nufus_future["yıl"].max()))
print("Nüfus il sayısı:", nufus_future["il_adı"].nunique())

# Beklenen satır kontrolü
beklenen = df["il_adı"].nunique() * len(GELECEK_YILLAR)
if len(nufus_future) != beklenen:
    print(f"[UYARI] Beklenen nüfus satırı {beklenen}, gelen satır {len(nufus_future)}")



========== 1) VERİ OKUMA ==========
Model verisi: (1863, 12)
İl sayısı: 81
Yıl aralığı: 2002 → 2024
Nüfus projeksiyonu: (486, 3)
Nüfus yıl aralığı: 2025 → 2030
Nüfus il sayısı: 81


In [39]:
# %% [5] SABİT ENSEMBLE TABLOSUNU HAZIRLA ===========================
print("\n========== 2) SABİT ENSEMBLE SEÇİMİ ==========")
secimler = pd.DataFrame(SABIT_SECIMLER)

# Her hedefte ağırlıkları tekrar normalize et. Yuvarlama farkı varsa sorun çıkmasın.
secimler["Agirlik"] = secimler.groupby("Hedef")["Agirlik"].transform(lambda s: s / s.sum())
secimler.to_csv("ensemble_secimleri_sabit.csv", index=False, encoding="utf-8-sig")
print(secimler.to_string(index=False))
print("[OK] ensemble_secimleri_sabit.csv")

agirlik_map = {
    hedef: dict(zip(grup["Model"], grup["Agirlik"]))
    for hedef, grup in secimler.groupby("Hedef")
}

# Kurulum kontrolü
kullanilan_modeller = sorted(secimler["Model"].unique())
if "CatBoost" in kullanilan_modeller and not HAVE_CAT:
    raise ImportError("Seçilen ensemble içinde CatBoost var ama catboost yüklü değil. Kurulum: pip install catboost")
if "XGBoost" in kullanilan_modeller and not HAVE_XGB:
    raise ImportError("Seçilen ensemble içinde XGBoost var ama xgboost yüklü değil. Kurulum: pip install xgboost")



========== 2) SABİT ENSEMBLE SEÇİMİ ==========
  Hedef                Model  Accuracy        MAE        RMSE  Agirlik
  diger                ARIMA 90.598394 375.372619 1146.702571 0.336859
  diger          HoltWinters 90.478957 379.220052 1206.847747 0.336415
  diger HistGradientBoosting 87.873100 607.133129 2742.472716 0.326726
 doktor          HoltWinters 91.975715 231.135847  685.106223 0.337337
 doktor                ARIMA 90.619329 300.440868  947.615696 0.332362
 doktor             CatBoost 90.057644 283.047003 1082.125630 0.330302
    ebe                ARIMA 93.776671  34.045168   68.273248 0.345603
    ebe          HoltWinters 92.300293  39.515053   64.821849 0.340162
    ebe         RandomForest 85.264967  82.050586  127.816354 0.314234
hastane                ARIMA 95.956145   0.654135    1.411002 0.356412
hastane          HoltWinters 93.197268   0.974772    1.572279 0.346165
hastane               Voting 80.074442   2.558210    5.837821 0.297423
hemsire           ExtraTrees 

In [41]:
# %% [6] 2025-2030 BASELINE OLUŞTUR ==================================
print("\n========== 3) 2025-2030 BASELINE OLUŞTURMA ==========")
iller = sorted(df["il_adı"].unique())
nf_map = {(r["il_adı"], int(r["yıl"])): float(r["nufus"]) for _, r in nufus_future.iterrows()}

# Nüfus dosyasında eksik il-yıl varsa son 10 yıl trendiyle doldur
lineer_doldu = []
for il in iller:
    seri = df[df["il_adı"] == il].dropna(subset=["nufus"]).tail(10)
    coef = np.polyfit(seri["yıl"], seri["nufus"], 1) if len(seri) >= 3 else None
    for y in GELECEK_YILLAR:
        if (il, y) not in nf_map:
            if coef is not None:
                nf_map[(il, y)] = float(np.polyval(coef, y))
            else:
                nf_map[(il, y)] = float(seri["nufus"].iloc[-1])
            lineer_doldu.append((il, y))

if lineer_doldu:
    print(f"[UYARI] Nüfus projeksiyonunda eksik olan {len(lineer_doldu)} il-yıl lineer trendle dolduruldu.")

nufus_kontrol = pd.DataFrame([
    {"il_adı": il, "yıl": y, "nufus": nf_map[(il, y)]}
    for il in iller
    for y in GELECEK_YILLAR
])
nufus_kontrol.to_csv("nufus_projeksiyonu_2025_2030.csv", index=False, encoding="utf-8-sig")

gelecek_satirlar = []
for il in iller:
    for y in GELECEK_YILLAR:
        gelecek_satirlar.append({
            "il_adı": il,
            "yıl": y,
            "nufus": nf_map[(il, y)],
            **{h: np.nan for h in HEDEFLER},
        })

gelecek_df = pd.DataFrame(gelecek_satirlar)
baz = pd.concat([df, gelecek_df], ignore_index=True).sort_values(["il_adı", "yıl"]).reset_index(drop=True)

# Gelecek yıllar için başlangıç değeri: son bilinen hedef değer ileri taşınır.
# Iteratif tahmin sırasında bu değerler model çıktısıyla güncellenecek.
for h in HEDEFLER:
    baz[h] = baz.groupby("il_adı")[h].ffill()

baz.to_csv("gelecek_baz_ensemble_sabit.csv", index=False, encoding="utf-8-sig")
print("Baseline:", baz.shape)
print("[OK] gelecek_baz_ensemble_sabit.csv")



========== 3) 2025-2030 BASELINE OLUŞTURMA ==========
Baseline: (2349, 12)
[OK] gelecek_baz_ensemble_sabit.csv


In [43]:
# %% [7] UNIVARIATE MODEL TAHMİNLERİ ================================
print("\n========== 4) UNIVARIATE MODEL TAHMİNLERİ ==========")
univar_pred = {}  # key: (il, hedef, model, yıl)

for hedef in HEDEFLER:
    for model_ad in agirlik_map[hedef].keys():
        if model_ad not in UNIVARIATE_MODELS:
            continue
        for il in iller:
            seri_il = df[df["il_adı"] == il].sort_values("yıl")
            tmp = seri_il.dropna(subset=[hedef])
            if tmp.empty:
                continue
            preds = univariate_forecast(model_ad, tmp["yıl"].values, tmp[hedef].values, h=len(GELECEK_YILLAR))
            for y, p in zip(GELECEK_YILLAR, preds):
                univar_pred[(il, hedef, model_ad, y)] = float(p)
        print(f"  {hedef:<10} - {model_ad:<12} hazır")


========== 4) UNIVARIATE MODEL TAHMİNLERİ ==========
  doktor     - HoltWinters  hazır
  doktor     - ARIMA        hazır
  ebe        - ARIMA        hazır
  ebe        - HoltWinters  hazır
  diger      - ARIMA        hazır
  diger      - HoltWinters  hazır
  yatak      - ARIMA        hazır
  yatak      - HoltWinters  hazır
  hastane    - ARIMA        hazır
  hastane    - HoltWinters  hazır


In [44]:
# %% [8] MULTIVARIATE SEÇİLMİŞ MODELLERİ FİT ET ======================
print("\n========== 5) SEÇİLMİŞ MULTIVARIATE MODELLERİ FİT ETME ==========")
trained = {}  # key: (hedef, model_ad)

for hedef in HEDEFLER:
    feats = FEATURE_MAP[hedef]
    check_required_columns(df, feats + [hedef], f"{hedef} eğitim verisi")
    train = df[["il_adı", "yıl"] + feats + [hedef]].dropna().copy()

    if train.empty:
        raise ValueError(f"{hedef} için eğitim verisi boş. Feature kolonlarını kontrol et: {feats}")

    X_tr = train[feats].values
    y_tr = train[hedef].values

    for model_ad in agirlik_map[hedef].keys():
        if model_ad in UNIVARIATE_MODELS:
            continue
        model = make_model(model_ad)
        model.fit(X_tr, y_tr)
        trained[(hedef, model_ad)] = model
        print(f"  {hedef:<10} - {model_ad:<22} fit edildi")



========== 5) SEÇİLMİŞ MULTIVARIATE MODELLERİ FİT ETME ==========
  doktor     - CatBoost               fit edildi
  ebe        - RandomForest           fit edildi
  diger      - HistGradientBoosting   fit edildi
  hemsire    - ExtraTrees             fit edildi
  hemsire    - Voting                 fit edildi
  hemsire    - XGBoost                fit edildi
  yatak      - CatBoost               fit edildi
  hastane    - Voting                 fit edildi


In [45]:
# %% [9] 2025-2030 ENSEMBLE TAHMİN ==================================
print("\n========== 6) 2025-2030 ENSEMBLE TAHMİN ==========")

for y in GELECEK_YILLAR:
    print(f"\n--- {y} tahmin ediliyor ---")

    # Aynı yılda hedefler birbirine feature olduğu için birkaç tur iterasyon yapılır.
    for tur in range(ITERASYON_TURU):
        for hedef in HEDEFLER:
            feats = FEATURE_MAP[hedef]
            weights = agirlik_map[hedef]

            for il in iller:
                idx_list = baz.index[(baz["il_adı"] == il) & (baz["yıl"] == y)]
                if len(idx_list) == 0:
                    continue
                idx = idx_list[0]

                model_preds = {}

                # ARIMA / HoltWinters tahminleri
                for model_ad in weights.keys():
                    if model_ad in UNIVARIATE_MODELS:
                        model_preds[model_ad] = univar_pred.get((il, hedef, model_ad, y), np.nan)

                # ML model tahminleri
                row = baz.loc[[idx], feats]
                if not row.isna().any(axis=1).iloc[0]:
                    X = row.values
                    for model_ad in weights.keys():
                        if model_ad in UNIVARIATE_MODELS:
                            continue
                        model = trained.get((hedef, model_ad))
                        if model is None:
                            continue
                        try:
                            model_preds[model_ad] = float(np.clip(model.predict(X)[0], 0, None))
                        except Exception:
                            model_preds[model_ad] = np.nan

                ens = weighted_average(model_preds, weights)
                if pd.notna(ens):
                    baz.loc[idx, hedef] = ens

    print(f"  {y} tamamlandı")



========== 6) 2025-2030 ENSEMBLE TAHMİN ==========

--- 2025 tahmin ediliyor ---
  2025 tamamlandı

--- 2026 tahmin ediliyor ---
  2026 tamamlandı

--- 2027 tahmin ediliyor ---
  2027 tamamlandı

--- 2028 tahmin ediliyor ---
  2028 tamamlandı

--- 2029 tahmin ediliyor ---
  2029 tamamlandı

--- 2030 tahmin ediliyor ---
  2030 tamamlandı


In [46]:
# %% [10] ÇIKTI =======================================================
print("\n========== 7) ÇIKTI ==========")

for h in HEDEFLER:
    baz.loc[baz["yıl"].isin(GELECEK_YILLAR), h] = np.clip(
        baz.loc[baz["yıl"].isin(GELECEK_YILLAR), h].astype(float), 0, None
    )

# Sayım değişkenleri tam sayıya yuvarlanır
for col in HEDEFLER:
    baz.loc[baz["yıl"].isin(GELECEK_YILLAR), col] = (
        baz.loc[baz["yıl"].isin(GELECEK_YILLAR), col].round().astype("Int64")
    )

sonuc = (
    baz[baz["yıl"].isin(GELECEK_YILLAR)]
    [["il_adı", "yıl", "nufus"] + HEDEFLER]
    .sort_values(["il_adı", "yıl"])
    .reset_index(drop=True)
)

eksik_sayisi = sonuc[["nufus"] + HEDEFLER].isna().sum()
print("Eksik değer kontrolü:")
print(eksik_sayisi.to_string())

sonuc.to_csv("tahmin_2025_2030_ensemble_sabit.csv", index=False, encoding="utf-8-sig")
sonuc.to_excel("tahmin_2025_2030_ensemble_sabit.xlsx", index=False)

print("\n[OK] tahmin_2025_2030_ensemble_sabit.csv")
print("[OK] tahmin_2025_2030_ensemble_sabit.xlsx")
print(f"Satır sayısı: {len(sonuc)} = {len(iller)} il × {len(GELECEK_YILLAR)} yıl")
print(f"Süre: {(time.time() - t0) / 60:.1f} dk")
print("\nİlk 12 satır:")
print(sonuc.head(12).to_string(index=False))



========== 7) ÇIKTI ==========
Eksik değer kontrolü:
nufus      0
doktor     0
ebe        0
diger      0
hemsire    0
yatak      0
hastane    0

[OK] tahmin_2025_2030_ensemble_sabit.csv
[OK] tahmin_2025_2030_ensemble_sabit.xlsx
Satır sayısı: 486 = 81 il × 6 yıl
Süre: 4.5 dk

İlk 12 satır:
  il_adı  yıl     nufus  doktor    ebe   diger  hemsire  yatak  hastane
   Adana 2025 2283609.0  5948.0 1735.0  8415.0   6921.0 7460.0     32.0
   Adana 2026 2293499.0  6082.0 1749.0  8949.0   6921.0 7528.0     33.0
   Adana 2027 2299727.0  6217.0 1762.0  9479.0   6921.0 7596.0     33.0
   Adana 2028 2305551.0  6352.0 1776.0 10007.0   6921.0 7663.0     33.0
   Adana 2029 2310973.0  6487.0 1790.0 10531.0   6921.0 7730.0     34.0
   Adana 2030 2315809.0  6621.0 1804.0 11052.0   6921.0 7796.0     34.0
Adıyaman 2025  617821.0   968.0  691.0  2229.0   1929.0 1098.0     11.0
Adıyaman 2026  623112.0   971.0  707.0  2342.0   1929.0 1106.0     11.0
Adıyaman 2027  629218.0   977.0  722.0  2455.0   1929.0 1115.